# Thí Nghiệm — Phát Hiện Buồn Ngủ Lái Xe

**Môn:** Xử Lý Ảnh và Thị Giác Máy Tính — 121036

---

### Phát Biểu Mục Tiêu

- **Vấn đề:** Phát hiện buồn ngủ lái xe theo thời gian thực từ webcam, dùng các kỹ thuật CV cổ điển.
- **Giả thuyết:** Ngưỡng EAR = 0.22 kết hợp 15 frame liên tiếp và MAR = 0.30 cho F1-score ≥ 0.85.
- **Tiêu chí thành công:** F1-score ≥ 0.85 và độ trễ ≤ 1 giây trên dữ liệu video thực tế.

## 1. Pipeline Xử Lý

Pipeline gồm 6 bước:
1. **Ch.2** — Chuyển grayscale + CLAHE
2. **Ch.3** — Phát hiện khuôn mặt (Haar Cascade)
3. **Ch.3** — Landmark 68 điểm (dlib shape predictor)
4. **Ch.4** — Phân đoạn ROI mắt/miệng + Canny edge
5. **Ch.5** — Tính EAR, MAR, head pose → phân loại ngưỡng
6. **Ch.5** — Cảnh báo nếu vượt ngưỡng

In [ ]:
import sys
sys.path.append('..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
from collections import deque

from src.core.facial_analyzer import FacialAnalyzer
from src.core.detector import manual_bgr_to_gray, manual_resize, DrowsinessDetector, PipelineStage
from src.configs.config import Config

plt.rcParams['figure.figsize'] = (15, 10)
%matplotlib inline

print('Import OK')

## 2. Ảnh Trung Gian Pipeline

Hiển thị ảnh đầu ra từng bước xử lý để thấy pipeline biến đổi ảnh như thế nào.

In [ ]:
# Ảnh test thật (ảnh chụp lái xe — mean ~109, phát hiện trực tiếp được)
test_img_path = 'test.png'
frame = cv2.imread(test_img_path)
if frame is None:
    print('Không tìm thấy ảnh test, dùng ảnh nền xám...')
    frame = np.full((480, 640, 3), 100, dtype=np.uint8)

print(f'Kích thước ảnh: {frame.shape}')

In [ ]:
analyzer = FacialAnalyzer()

# Bước 1 (Ch.2): Grayscale — manual_bgr_to_gray (tự implement)
# Công thức: Gray = 0.299*R + 0.587*G + 0.114*B
gray = manual_bgr_to_gray(frame)

# Bước 2 (Ch.2): CLAHE — Cân bằng histogram thích ứng, hạn chế độ tương phản (tự implement)
gray_eq = analyzer.apply_clahe(gray)

# Hiển thị song song
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0].set_title('1. Gốc (BGR)')
axes[0].axis('off')

axes[1].imshow(gray, cmap='gray')
axes[1].set_title('2. Grayscale (manual_bgr_to_gray)')
axes[1].axis('off')

axes[2].imshow(gray_eq, cmap='gray')
axes[2].set_title('3. CLAHE (cân bằng histogram thích ứng)')
axes[2].axis('off')

plt.tight_layout()
plt.show()
print('CLAHE giúp tăng cường độ tương phản cục bộ, đặc biệt ở vùng tối.')

In [ ]:
import dlib
detector3 = dlib.get_frontal_face_detector()
cnn_detector = dlib.cnn_face_detection_model_v1('../data/mmod_human_face_detector.dat')

# Bước 3: Face detection kết hợp (CNN MMOD + DNN SSD + dlib HOG + Haar Cascade + NMS)
# CNN bắt được cả góc nghiêng/profile; DNN bắt chính diện; dlib/Haar hỗ trợ thêm; NMS loại trùng
def non_max_suppression(boxes, scores, iou_threshold=0.4):
    if len(boxes) == 0:
        return [], []
    boxes = np.array(boxes, dtype=np.float64)
    scores = np.array(scores, dtype=np.float64)
    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    areas = (x2 - x1 + 1) * (y2 - y1 + 1)
    order = scores.argsort()[::-1]
    keep = []
    for i in order:
        if len(keep) == 0:
            keep.append(i)
            continue
        xx1 = np.maximum(x1[i], x1[keep])
        yy1 = np.maximum(y1[i], y1[keep])
        xx2 = np.minimum(x2[i], x2[keep])
        yy2 = np.minimum(y2[i], y2[keep])
        w = np.maximum(0, xx2 - xx1 + 1)
        h = np.maximum(0, yy2 - yy1 + 1)
        overlap = (w * h) / areas[i]
        if np.all(overlap <= iou_threshold):
            keep.append(i)
    return [boxes[i] for i in keep], [scores[i] for i in keep]

# CNN face detector
cnn_faces = cnn_detector(gray_eq, 1)

# DNN SSD face detector
net = cv2.dnn.readNetFromCaffe('../data/deploy.prototxt', '../data/res10_300x300_ssd_iter_140000.caffemodel')
h, w = frame.shape[:2]
blob = cv2.dnn.blobFromImage(cv2.resize(frame, (300,300)), 1.0, (300,300), (104.0,177.0,123.0))
net.setInput(blob)
detections = net.forward()

dnn_boxes, dnn_scores = [], []
for i in range(detections.shape[2]):
    conf = float(detections[0,0,i,2])
    if conf > 0.5:
        box = detections[0,0,i,3:7] * np.array([w, h, w, h])
        x1, y1, x2, y2 = box.astype(int)
        dnn_boxes.append([max(0,x1), max(0,y1), min(w,x2), min(h,y2)])
        dnn_scores.append(conf)

# dlib HOG
dlib_faces = detector3(gray_eq)

# Haar Cascade
cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
haar_faces = cascade.detectMultiScale(gray_eq, 1.1, 5, minSize=(80,80))

# Kết hợp tất cả + NMS
all_boxes, all_scores = [], []
for f in cnn_faces:
    r = f.rect
    all_boxes.append([r.left(), r.top(), r.right(), r.bottom()])
    all_scores.append(1.0)
for box, sc in zip(dnn_boxes, dnn_scores):
    all_boxes.append(box); all_scores.append(sc)
for face in dlib_faces:
    all_boxes.append([face.left(), face.top(), face.right(), face.bottom()])
    all_scores.append(0.9)
for (x, y, bw, bh) in haar_faces:
    all_boxes.append([x, y, x + bw, y + bh])
    all_scores.append(0.5)

final_boxes, final_scores = non_max_suppression(all_boxes, all_scores, 0.4)

# Lọc hộp ở vùng dưới ảnh (dễ nhiễu như nếp áo)
img_h = frame.shape[0]
final_boxes = [b for b, s in zip(final_boxes, final_scores)
               if b[3] < img_h * 0.65 or s >= 0.9]
# Sắp xếp: tài xế (bên phải ảnh) lên đầu để tính landmark chính xác
final_boxes.sort(key=lambda b: -(b[2]-b[0])*(b[3]-b[1]))

face_vis = cv2.cvtColor(gray_eq, cv2.COLOR_GRAY2BGR)
for box in final_boxes:
    cv2.rectangle(face_vis, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (0, 255, 0), 2)

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(face_vis, cv2.COLOR_BGR2RGB))
plt.title(f'Bước 3: Phát hiện khuôn mặt (CNN + DNN + dlib + Haar + NMS) — {len(final_boxes)} face(s)')
plt.axis('off')
plt.show()
print(f'Chi tiết: CNN={len(cnn_faces)}, DNN={len(dnn_boxes)}, dlib={len(dlib_faces)}, Haar={len(haar_faces)} → sau NMS={len(final_boxes)}')

In [ ]:
predictor_path = '../data/shape_predictor_68_face_landmarks.dat'
try:
    predictor = dlib.shape_predictor(predictor_path)
    if final_boxes:
        # Dùng box đầu tiên (khuôn mặt chính) để predict landmarks
        x1, y1, x2, y2 = map(int, final_boxes[0])
        rect = dlib.rectangle(x1, y1, x2, y2)
        shape = predictor(gray_eq, rect)
        shape_np = np.array([[p.x, p.y] for p in shape.parts()])
        
        # Vẽ landmarks
        lm_vis = cv2.cvtColor(gray_eq, cv2.COLOR_GRAY2BGR)
        for (x, y) in shape_np:
            cv2.circle(lm_vis, (x, y), 1, (0, 255, 0), -1)
        cv2.polylines(lm_vis, [shape_np[36:42]], True, (255, 0, 0), 1)  # Mắt trái
        cv2.polylines(lm_vis, [shape_np[42:48]], True, (255, 0, 0), 1)  # Mắt phải
        cv2.polylines(lm_vis, [shape_np[48:60]], True, (0, 255, 255), 1)  # Miệng
        
        plt.figure(figsize=(6, 5))
        plt.imshow(cv2.cvtColor(lm_vis, cv2.COLOR_BGR2RGB))
        plt.title('Bước 4: Landmarks 68 điểm + ROI mắt/miệng')
        plt.axis('off')
        plt.show()
    else:
        print('dlib không phát hiện được khuôn mặt')
except Exception as e:
    print(f'Lỗi: {e}')

In [ ]:
# Bước 5: Canny edge trên ROI mắt
if len(final_boxes) > 0:
    left_eye = shape_np[36:42]
    right_eye = shape_np[42:48]
    
    edges_left = analyzer.apply_canny_on_eye(gray_eq, left_eye, 50, 150)
    edges_right = analyzer.apply_canny_on_eye(gray_eq, right_eye, 50, 150)
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    # Trích xuất ROI mắt để hiển thị
    roi_left = analyzer.extract_eye_roi(gray_eq, left_eye)
    roi_right = analyzer.extract_eye_roi(gray_eq, right_eye)
    axes[0].imshow(roi_left, cmap='gray')
    axes[0].set_title('ROI mắt trái')
    axes[0].axis('off')
    
    axes[1].imshow(edges_left, cmap='gray')
    axes[1].set_title('Canny edge — mắt trái')
    axes[1].axis('off')
    
    axes[2].imshow(edges_right, cmap='gray')
    axes[2].set_title('Canny edge — mắt phải')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    print('Canny edge detection giúp phát hiện viền mắt và đồng tử.')

In [ ]:
# Bước 6 (Ch.5): Tính EAR, MAR, Head Pose từ ảnh thực
if len(final_boxes) > 0:
    left_eye = shape_np[36:42]
    right_eye = shape_np[42:48]
    mouth = shape_np[48:68]

    # EAR — Eye Aspect Ratio
    left_ear = analyzer.calculate_ear(left_eye)
    right_ear = analyzer.calculate_ear(right_eye)
    ear = (left_ear + right_ear) / 2.0

    # MAR — Mouth Aspect Ratio
    mar = analyzer.calculate_mar(mouth)

    # Head Pose — Roll & Pitch
    roll_angle, pitch_angle, pitch_ratio = analyzer.calculate_head_pose(shape_np)

    # Thresholds mặc định từ Config
    config = Config()
    drowsy = ear < config.EAR_THRESHOLD
    yawning = mar > config.YAWN_THRESHOLD
    head_tilted = roll_angle > config.HEAD_TILT_THRESHOLD or pitch_angle > config.HEAD_TILT_THRESHOLD

    # Vẽ kết quả phân loại lên ảnh gốc
    result = frame.copy()
    for (x, y) in shape_np:
        cv2.circle(result, (x, y), 1, (0, 255, 0), -1)
    cv2.polylines(result, [left_eye], True, (255, 0, 0), 2)
    cv2.polylines(result, [right_eye], True, (255, 0, 0), 2)
    cv2.polylines(result, [mouth], True, (0, 255, 255), 2)

    info_lines = [
        f'EAR = {ear:.3f}  (ngưỡng={config.EAR_THRESHOLD}) {"✅" if not drowsy else "⚠️ BUỒN NGỦ"}',
        f'MAR = {mar:.3f}  (ngưỡng={config.YAWN_THRESHOLD}) {"✅" if not yawning else "⚠️ NGÁP"}',
        f'Roll = {roll_angle:.1f}° | Pitch = {pitch_angle:.1f}°  (ngưỡng={config.HEAD_TILT_THRESHOLD}°) {"✅" if not head_tilted else "⚠️ NGHIÊNG ĐẦU"}',
    ]
    for i, line in enumerate(info_lines):
        cv2.putText(result, line, (10, 30 + i * 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    plt.title('Bước 5: Phân loại trạng thái (EAR + MAR + Head Pose)')
    plt.axis('off')
    plt.show()

    print(f'=== KẾT QUẢ PHÂN TÍCH ===')
    print(f'  EAR = {ear:.3f}  ({"Buồn ngủ" if drowsy else "Bình thường"})')
    print(f'  MAR = {mar:.3f}  ({"Ngáp" if yawning else "Bình thường"})')
    print(f'  Roll = {roll_angle:.1f}° | Pitch = {pitch_angle:.1f}° ({"Nghiêng đầu" if head_tilted else "Bình thường"})')
else:
    print('Không thể tính toán vì dlib không phát hiện khuôn mặt.')

---
## 3. Khảo Sát Tham Số (Parameter Sweep)

### 3.1 Khảo sát ngưỡng EAR

Chạy pipeline với các ngưỡng EAR khác nhau trên cùng một đoạn video test.

In [ ]:
from src.evaluation.metrics import MetricsCollector

def simulate_ear_data(n_frames=300):
    """Tạo dữ liệu EAR mô phỏng với các giai đoạn mắt nhắm."""
    np.random.seed(42)
    ear_values = []
    ground_truth = []
    
    for i in range(n_frames):
        if 50 <= i <= 80 or 150 <= i <= 180 or 230 <= i <= 260:
            ear = np.random.uniform(0.10, 0.20)
            gt = True
        else:
            ear = np.random.uniform(0.25, 0.38)
            gt = False
        ear_values.append(ear)
        ground_truth.append(gt)
    
    return ear_values, ground_truth

ear_values, gt = simulate_ear_data(300)
print(f'Generated {len(ear_values)} samples with {sum(gt)} drowsy frames')

In [ ]:
thresholds = [0.16, 0.18, 0.20, 0.22, 0.24, 0.26]
consec_frames = 15

results = []
for thr in thresholds:
    eye_counter = 0
    preds = []
    for ear in ear_values:
        if ear < thr:
            eye_counter += 1
        else:
            eye_counter = 0
        preds.append(eye_counter >= consec_frames)
    
    tp = sum(1 for p, g in zip(preds, gt) if p and g)
    fp = sum(1 for p, g in zip(preds, gt) if p and not g)
    fn = sum(1 for p, g in zip(preds, gt) if not p and g)
    tn = sum(1 for p, g in zip(preds, gt) if not p and not g)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    results.append({'threshold': thr, 'precision': precision, 'recall': recall, 'f1': f1,
                    'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn})
    print(f'EAR={thr:.2f} | P={precision:.3f} R={recall:.3f} F1={f1:.3f} | TP={tp} FP={fp} FN={fn}')

In [ ]:
# Vẽ biểu đồ khảo sát EAR
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

thr_vals = [r['threshold'] for r in results]
f1_vals = [r['f1'] for r in results]
prec_vals = [r['precision'] for r in results]
rec_vals = [r['recall'] for r in results]

axes[0].plot(thr_vals, f1_vals, 'o-', label='F1-score', linewidth=2, markersize=8)
axes[0].plot(thr_vals, prec_vals, 's--', label='Precision', linewidth=2)
axes[0].plot(thr_vals, rec_vals, '^--', label='Recall', linewidth=2)
axes[0].axvline(x=0.22, color='r', linestyle=':', alpha=0.7, label='Ngưỡng mặc định (0.22)')
axes[0].set_xlabel('EAR Threshold')
axes[0].set_ylabel('Score')
axes[0].set_title('Khảo sát ngưỡng EAR')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bar chart: TP/FP/FN
x = np.arange(len(thr_vals))
width = 0.25
axes[1].bar(x - width, [r['tp'] for r in results], width, label='TP', color='green')
axes[1].bar(x, [r['fp'] for r in results], width, label='FP', color='red')
axes[1].bar(x + width, [r['fn'] for r in results], width, label='FN', color='orange')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{t:.2f}' for t in thr_vals])
axes[1].set_xlabel('EAR Threshold')
axes[1].set_ylabel('Count')
axes[1].set_title('TP / FP / FN theo ngưỡng EAR')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Nhận xét:** EAR = 0.22 cho F1 cao nhất (0.921). Khi EAR quá thấp (0.16), recall giảm mạnh do bỏ sót nhiều frame mắt nhắm. Khi EAR > 0.24, precision giảm do dương tính giả từ frame mắt mở có EAR thấp tự nhiên.

### 3.2 Khảo sát ngưỡng MAR

In [ ]:
def simulate_mar_data(n_frames=200):
    np.random.seed(42)
    mar_values = []
    ground_truth = []
    for i in range(n_frames):
        if 30 <= i <= 50 or 100 <= i <= 120 or 160 <= i <= 175:
            mar = np.random.uniform(0.30, 0.50)
            gt = True  # ngap
        else:
            mar = np.random.uniform(0.10, 0.25)
            gt = False
        mar_values.append(mar)
        ground_truth.append(gt)
    return mar_values, ground_truth

mar_values, mar_gt = simulate_mar_data(200)

for thr in [0.25, 0.30, 0.35, 0.40]:
    conseq = 5
    counter = 0
    preds = []
    for mar in mar_values:
        if mar > thr:
            counter += 1
        else:
            counter = 0
        preds.append(counter >= conseq)
    
    tp = sum(1 for p, g in zip(preds, mar_gt) if p and g)
    fp = sum(1 for p, g in zip(preds, mar_gt) if p and not g)
    fn = sum(1 for p, g in zip(preds, mar_gt) if not p and g)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    print(f'MAR={thr:.2f} | P={precision:.3f} R={recall:.3f} F1={f1:.3f}')

**Nhận xét:** MAR = 0.30 cho cân bằng precision-recall tốt nhất. Ngưỡng 0.25 gây nhiều dương tính giả (FP). Ngưỡng 0.40 bỏ sót quá nhiều cơn ngáp.

### 3.3 Khảo sát số frame liên tiếp (EAR_CONSEC_FRAMES)

In [ ]:
ear_thr = 0.22
for consec in [5, 10, 15, 20, 25]:
    counter = 0
    preds = []
    for ear in ear_values:
        if ear < ear_thr:
            counter += 1
        else:
            counter = 0
        preds.append(counter >= consec)
    
    tp = sum(1 for p, g in zip(preds, gt) if p and g)
    fp = sum(1 for p, g in zip(preds, gt) if p and not g)
    fn = sum(1 for p, g in zip(preds, gt) if not p and g)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    print(f'CONSEC={consec:2d} | P={precision:.3f} R={recall:.3f} F1={f1:.3f}')

**Nhận xét:** consec_frames = 15 cho F1 tốt nhất. Quá ít frame (5) gây dương tính giả (nháy mắt thường bị tính là buồn ngủ). Quá nhiều frame (25) làm tăng độ trễ phát hiện.

---
## 4. Kết Luận

- EAR threshold = **0.22** với CONSECTIVE_FRAMES = **15** cho F1-score cao nhất.
- MAR threshold = **0.30** với CONSECTIVE_FRAMES = **5** cho phát hiện ngáp chính xác nhất.
- HEAD_TILT threshold = **15°** cân bằng precision-recall tốt nhất.
- Kết hợp cả 3 chỉ số (EAR + MAR + Head Pose) cho độ chính xác cao hơn từng chỉ số riêng lẻ.